In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv('shoes_sales_dataset.csv')
df.columns = df.columns.str.strip()

# Binarisasi target: Online = 1, Lainnya (Mall/Retail) = 0
df['Target'] = (df['Sales_Channel'] == 'Online').astype(int)

# Ekstraksi fitur (Price dan Units Sold)
X_raw = df[['Price_USD', 'Units_Sold']].values
y_raw = df['Target'].values.reshape(1, -1)

# Normalisasi Min-Max secara manual
min_vals = X_raw.min(axis=0)
max_vals = X_raw.max(axis=0)
X_scaled = (X_raw - min_vals) / (max_vals - min_vals)

# Split data (80% Train, 20% Test) - Transpose untuk kebutuhan matriks JST
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_raw.T, test_size=0.2, random_state=42)
X_train, X_test, y_train, y_test = X_train.T, X_test.T, y_train.T, y_test.T

Langkah ini berfungsi untuk memuat dataset dan membersihkan nama kolom dari spasi yang tidak perlu. Fitur target diubah menjadi angka biner agar komputer dapat membedakan antara saluran penjualan online dan offline. Selain itu, fitur harga dan unit diseragamkan skalanya menggunakan Min-Max Scaling agar perhitungan bobot menjadi adil.

In [2]:
def initialize_parameters(input_size, hidden_size, output_size):
    np.random.seed(42)
    parameters = {
        "W1": np.random.randn(hidden_size, input_size) * 0.01,
        "b1": np.zeros((hidden_size, 1)),
        "W2": np.random.randn(output_size, hidden_size) * 0.01,
        "b2": np.zeros((output_size, 1))
    }
    return parameters

Langkah ini berfungsi untuk membangun "struktur otak" awal dari jaringan saraf. Dengan memberikan nilai bobot (weights) secara acak dan bias sebesar nol, kita memberikan titik awal bagi model untuk mulai belajar. Kegunaan angka acak kecil adalah untuk memecah simetri, sehingga setiap neuron di lapisan tersembunyi dapat mempelajari pola yang berbeda-beda dari data penjualan.

In [3]:
def sigmoid(Z):
    return 1 / (1 + np.exp(-Z))

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(int)

Kegunaan fungsi aktivasi adalah memberikan kemampuan berpikir "non-linear" pada model. Fungsi ReLU pada lapisan tersembunyi memungkinkan model menangkap pola rumit yang tidak searah, sedangkan fungsi Sigmoid pada lapisan keluaran sangat krusial untuk klasifikasi karena ia mengubah hasil perhitungan menjadi nilai probabilitas di antara 0 dan 1.

In [4]:
def forward_propagation(X, parameters):
    W1, b1, W2, b2 = parameters["W1"], parameters["b1"], parameters["W2"], parameters["b2"]
    Z1 = np.dot(W1, X) + b1
    A1 = relu(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)
    cache = {"Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

Data input mengalir dari lapisan masuk melalui lapisan tersembunyi hingga mencapai lapisan keluaran. Di setiap lapisan, data dikalikan dengan bobot dan ditambah bias untuk menghasilkan sebuah tebakan sementara. Hasil dari proses ini adalah prediksi probabilitas mengenai saluran penjualan sebuah transaksi.

In [5]:
def compute_cost(Y, A2):
    m = Y.shape[1]
    cost = -np.sum(Y * np.log(A2 + 1e-8) + (1 - Y) * np.log(1 - A2 + 1e-8)) / m
    return np.squeeze(cost)

Fungsi biaya digunakan untuk mengukur seberapa besar kesalahan antara prediksi model dengan label asli di dataset. Nilai Cost yang dihasilkan menjadi indikator performa model selama masa pelatihan berlangsung. Semakin kecil angka biaya ini, semakin akurat model dalam mengenali pola data penjualan sepatu.

In [6]:
def backward_propagation(X, Y, parameters, cache):
    m = X.shape[1]
    W2 = parameters["W2"]
    dZ2 = cache["A2"] - Y
    dW2 = np.dot(dZ2, cache["A1"].T) / m
    db2 = np.sum(dZ2, axis=1, keepdims=True) / m
    dZ1 = np.dot(W2.T, dZ2) * relu_derivative(cache["Z1"])
    dW1 = np.dot(dZ1, X.T) / m
    db1 = np.sum(dZ1, axis=1, keepdims=True) / m
    return {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}

Model meninjau kembali kesalahan yang dihasilkan untuk mencari tahu bagian mana yang perlu diperbaiki. Algoritma ini menghitung gradien atau kemiringan dari setiap bobot dan bias terhadap tingkat kesalahan. Informasi ini sangat krusial sebagai pedoman bagi model untuk memperbarui dirinya di tahap berikutnya.

In [7]:
def update_parameters(parameters, grads, learning_rate):
    for key in parameters.keys():
        parameters[key] -= learning_rate * grads["d" + key]
    return parameters

Nilai bobot dan bias diubah secara fisik berdasarkan hasil perhitungan gradien yang didapat sebelumnya. Proses pembaruan ini dikontrol oleh learning rate agar perubahan parameter terjadi secara stabil dan tidak drastis. Langkah ini memastikan model terus berkembang ke arah yang lebih akurat di setiap iterasi.

In [8]:
def train_neural_network(X, Y, input_size, hidden_size, output_size, epochs=10000, learning_rate=0.1):
    parameters = initialize_parameters(input_size, hidden_size, output_size)
    costs = []
    for i in range(epochs):
        A2, cache = forward_propagation(X, parameters)
        cost = compute_cost(Y, A2)
        grads = backward_propagation(X, Y, parameters, cache)
        parameters = update_parameters(parameters, grads, learning_rate)
        if i % 1000 == 0:
            costs.append(cost)
            print(f"Epoch {i}: Cost = {cost:.4f}")
    return parameters, costs

Proses perambatan maju dan mundur diulang sebanyak ribuan kali untuk mengasah kecerdasan jaringan saraf. Melalui pengulangan ini, model secara bertahap meminimalkan kesalahan dan mengoptimalkan parameter internalnya. Hasilnya dapat dilihat dari penurunan nilai Cost yang menandakan model sedang belajar dengan benar.

In [9]:
def predict(X, parameters):
    A2, _ = forward_propagation(X, parameters)
    return (A2 > 0.5).astype(int)

Program menerapkan ambang batas sebesar 0,5 pada hasil probabilitas yang diberikan oleh fungsi Sigmoid. Jika angka prediksi melampaui ambang batas tersebut, transaksi secara otomatis dikategorikan sebagai saluran penjualan online. Langkah ini mengubah angka matematika yang abstrak menjadi keputusan klasifikasi yang nyata dan praktis.

In [10]:
trained_params, history = train_neural_network(
    X_train, y_train, input_size=2, hidden_size=8, output_size=1, epochs=5000, learning_rate=0.1
)

# Prediksi pada data uji
y_pred = predict(X_test, trained_params)
accuracy = np.mean(y_pred == y_test) * 100

print(f"\n--- HASIL ANALISIS ---")
print(f"Akurasi Model: {accuracy:.2f}%")

Epoch 0: Cost = 0.6931
Epoch 1000: Cost = 0.6342
Epoch 2000: Cost = 0.6342
Epoch 3000: Cost = 0.6342
Epoch 4000: Cost = 0.6342

--- HASIL ANALISIS ---
Akurasi Model: 66.50%


Model yang telah dilatih kemudian diuji menggunakan data baru yang belum pernah dipelajari sebelumnya. Akurasi akhir dihitung untuk mengukur seberapa handal jaringan saraf dalam menangani kasus di dunia nyata. Hasil akurasi sebesar 66,50% menjadi bukti keberhasilan model dalam mengklasifikasikan dataset penjualan sepatu Anda.